# 🎯 DeClickBait: TigerLLM 1B Bangla LoRA Fine-Tuning
### Multilingual Clickbait Detection & Headline De-biasing
**Model**: `md-nishat-008/TigerLLM-1B-it`  
**Architecture**: `LLM`  
**Target Environments**: Google Colab (Free T4) & Kaggle (Dual T4 $\times$ 2 / P100 / CPU)

> **Overview**: Fine-tunes TigerLLM 1B (Instruct) with LoRA. Continually pre-trained on Bangla-TextBook corpus and fine-tuned on Bangla-Instruct for Bengali NLP.


## 1. Environment Setup & Dependency Installation


In [ ]:
# Install required libraries quietly
!pip install -q transformers>=4.44.0 peft>=0.12.0 accelerate>=0.33.0 bitsandbytes>=0.43.0 datasets>=2.20.0 evaluate>=0.4.2 rouge-score>=0.1.2 sentencepiece>=0.2.0 scikit-learn pandas numpy matplotlib seaborn onnx onnxruntime
print("✓ Environment dependencies installed successfully!")


## 2. Imports & Cross-Platform Hardware Profiling


In [ ]:
import os
import sys
import gc
import json
import shutil
import random
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

# Set seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Detect Runtime Environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/working")
ENV_NAME = "Google Colab" if IS_COLAB else ("Kaggle" if IS_KAGGLE else "Local/Server")

gpu_count = torch.cuda.device_count()
device_name = torch.cuda.get_device_name(0) if gpu_count > 0 else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if gpu_count > 0 else 0.0

# Precision configuration
PRECISION = "fp16" if torch.cuda.is_available() else "fp32"
OUTPUT_DIR = "./outputs_tigerllm_1b_lora"

print("=" * 60)
print(f"🌍 Environment:        {ENV_NAME}")
print(f"🎮 Compute Device:     {device_name} ({gpu_count} GPU(s) available)")
if gpu_count > 0:
    print(f"💾 GPU VRAM:           {vram_gb:.2f} GB")
if gpu_count > 1:
    print(f"⚡ Multi-GPU Detected! Kaggle Dual T4x2 load balancing active.")
print(f"🎯 Target Output Dir:   {OUTPUT_DIR}")
print("=" * 60)


## 3. Checkpoint & Resumption Helpers


In [ ]:
def find_latest_checkpoint(dir_path: str | Path) -> Optional[str]:
    """Finds the most recent checkpoint-XXXX directory inside dir_path."""
    p = Path(dir_path)
    if not p.exists():
        return None
    ckpts = [d for d in p.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
    if not ckpts:
        return None

    def get_step(d: Path) -> int:
        try:
            return int(d.name.split("-")[-1])
        except ValueError:
            return 0

    ckpts.sort(key=get_step)
    return str(ckpts[-1])

def mount_google_drive(mount_point: str = "/content/drive") -> bool:
    """Mounts Google Drive on Colab if desired."""
    if "google.colab" in sys.modules:
        try:
            from google.colab import drive
            drive.mount(mount_point)
            print(f"✓ Google Drive mounted at {mount_point}")
            return True
        except Exception as e:
            print(f"Google Drive notice: {e}")
            return False
    return False


## 4. Dataset Ingestion (100% Constant Persistent Splits)
- **Identity Mapping Rule**: Non-clickbait targets remain identical to `original_title` to prevent corrupting factual headlines.
- **Persistent Splitting**: Splits are persisted to `splits/` so train/val/test records remain **100% constant and identical** across all runs, machines, Colab, and Kaggle.


In [ ]:
def load_and_preprocess_corpus(
    corpus_path: Optional[str] = None,
    splits_dir: str = "splits",
    split_mode: str = "train_val_test", # 80/10/10
    max_samples: Optional[int] = None,
    force_recreate_splits: bool = False,
    seed: int = 42
) -> Dict[str, Any]:
    """Loads balanced corpus with persistent constant splits."""
    s_dir = Path(splits_dir)
    s_dir.mkdir(parents=True, exist_ok=True)
    train_file = s_dir / f"train_{split_mode}.jsonl"
    val_file = s_dir / f"val_{split_mode}.jsonl"
    test_file = s_dir / f"test_{split_mode}.jsonl"

    candidates = [
        Path(corpus_path) if corpus_path else None,
        Path("corpus_balanced.jsonl"),
        Path("data/corpus_balanced.jsonl"),
        Path("data/prepared/corpus_balanced.jsonl"),
        Path("/content/corpus_balanced.jsonl"),
        Path("/content/data/corpus_balanced.jsonl"),
        Path("/kaggle/input/declickbait/corpus_balanced.jsonl"),
        Path("/kaggle/working/corpus_balanced.jsonl"),
        Path("/kaggle/working/data/corpus_balanced.jsonl"),
    ]
    resolved_path = None
    for p in candidates:
        if p and p.exists():
            resolved_path = p
            break


    # 1. Check for existing persistent splits
    if not force_recreate_splits and train_file.exists() and val_file.exists():
        df_train_cached = pd.read_json(train_file, lines=True)
        # If cached split was from the small 3.2k fallback (<= 2560 train) and a real 25k corpus is found:
        if resolved_path and resolved_path.stat().st_size > 5 * 1024 * 1024 and len(df_train_cached) <= 2560:
            print(f"🔄 Detected real full corpus at {resolved_path} ({resolved_path.stat().st_size / (1024**2):.1f} MB)! Upgrading splits from previous 3.2k fallback...")
        else:
            print(f"📦 Loading persistent constant dataset splits from: {s_dir}")
            df_val = pd.read_json(val_file, lines=True)
            df_test = pd.read_json(test_file, lines=True) if test_file.exists() else df_val.copy()
            if max_samples and len(df_train_cached) > max_samples:
                df_train_cached = df_train_cached.head(max_samples)
                df_val = df_val.head(max(10, int(max_samples * 0.1)))
                df_test = df_test.head(max(10, int(max_samples * 0.1)))
            print(f"✓ Loaded constant splits: Train={len(df_train_cached):,} | Val={len(df_val):,} | Test={len(df_test):,}")
            return {"df_train": df_train_cached, "df_val": df_val, "df_test": df_test}

    if resolved_path is None:
        print("⚠️ corpus_balanced.jsonl not found in working directory or Colab/Kaggle standard paths.")
        print("   Generating self-contained 3,200 sample pool fallback for demo run...")
        Path("data").mkdir(exist_ok=True)
        resolved_path = Path("data/corpus_balanced.jsonl")
        sample_pool = [
            {"original_title": "এরপর যা ঘটল তা জানলে আপনার মাথা ঘুরে যাবে!", "actual_title": "সচিবালয়ে নতুন বাজেট সংক্রান্ত বৈঠক সম্পন্ন", "body": "গতকাল সচিবালয়ে এক দীর্ঘ বৈঠক অনুষ্ঠিত হয় যেখানে আগামী অর্থবছরের বাজেট প্রস্তাবসমূহ অনুমোদন করা হয়েছে।", "language_mix": "bangla", "clickbait": True},
            {"original_title": "Ajke raate ja hote jacche ta shune shobai shock!", "actual_title": "Ajke raate shohore sholpo brishti hote pare", "body": "Abohawa office janacche je shohore ajke raate halka theke majhari brishti paat hote pare.", "language_mix": "banglish", "clickbait": True},
            {"original_title": "You won't believe what this celebrity looked like yesterday!", "actual_title": "Actor John Doe attends annual film awards gala in New York", "body": "Actor John Doe walked the red carpet yesterday during the 2026 Film Awards held in downtown Manhattan.", "language_mix": "english", "clickbait": True},
            {"original_title": "Chomok! Notun smartphone er price shune shobai obak", "actual_title": "XYZ Company announces flagship smartphone starting at $499", "body": "XYZ Company officially released their new flagship device with updated camera sensors and display.", "language_mix": "mixed", "clickbait": True},
            {"original_title": "প্রধানমন্ত্রীর নতুন অর্থনৈতিক পরিকল্পনার বিস্তারিত প্রকাশ", "actual_title": "প্রধানমন্ত্রীর নতুন অর্থনৈতিক পরিকল্পনার বিস্তারিত প্রকাশ", "body": "অর্থনৈতিক সমৃদ্ধি অর্জনের লক্ষ্যে ৫ বছর মেয়াদী বিশেষ উন্নয়ন প্যাকেজ ঘোষণা করেছেন প্রধানমন্ত্রী।", "language_mix": "bangla", "clickbait": False},
            {"original_title": "Dhaka metro rail schedule revised for holiday season", "actual_title": "Dhaka metro rail schedule revised for holiday season", "body": "Dhaka Mass Transit authorities have published the revised operational timetable for the festive period.", "language_mix": "english", "clickbait": False},
            {"original_title": "Shobai ke janano jacche je shorkari chuti shuru hocche", "actual_title": "Shobai ke janano jacche je shorkari chuti shuru hocche", "body": "Shorkari proggapon onujayi agami shoptaho theke shorkari chuti shuru hobe.", "language_mix": "banglish", "clickbait": False},
            {"original_title": "City corporation starts new road repair initiative", "actual_title": "City corporation starts new road repair initiative", "body": "Mayor announced the commencement of thoroughfare rehabilitation across key commercial corridors.", "language_mix": "mixed", "clickbait": False},
        ] * 400
        with open(resolved_path, "w", encoding="utf-8") as f:
            for item in sample_pool:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    records = []
    with open(resolved_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            orig = (r.get("original_title") or r.get("title") or "").strip()
            is_cb = bool(r.get("clickbait", False))
            target = (r.get("actual_title") or r.get("generated_title") or r.get("title_debiased") or "").strip() if is_cb else orig
            if orig and target:
                records.append({
                    "original_title": orig,
                    "target_title": target,
                    "body": (r.get("body") or r.get("transcript") or "").strip(),
                    "language_mix": r.get("language_mix") or r.get("language") or "mixed",
                    "clickbait": 1 if is_cb else 0
                })

    df = pd.DataFrame(records)
    from sklearn.model_selection import train_test_split
    df_train, df_temp = train_test_split(df, test_size=0.20, random_state=seed, stratify=df["clickbait"])
    df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=seed, stratify=df_temp["clickbait"])

    df_train.to_json(train_file, orient="records", lines=True, force_ascii=False)
    df_val.to_json(val_file, orient="records", lines=True, force_ascii=False)
    df_test.to_json(test_file, orient="records", lines=True, force_ascii=False)
    print(f"💾 Persistent constant splits saved to {s_dir} (Train={len(df_train):,}, Val={len(df_val):,}, Test={len(df_test):,})")

    return {"df_train": df_train.reset_index(drop=True), "df_val": df_val.reset_index(drop=True), "df_test": df_test.reset_index(drop=True)}

# Load dataset (Set max_samples=None to train on all 25,000 samples)
data_bundle = load_and_preprocess_corpus(split_mode="train_val_test", max_samples=None)


## 5. Tokenization & Subword Diagnostics


In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "md-nishat-008/TigerLLM-1B-it"
print(f"Inspecting tokenizer for {MODEL_ID}...")
if "llm" != "multihead":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    test_phrases = [
        ("Bangla Script", "আজকে চরম এক চাঞ্চল্যকর খবর ফাঁস হলো"),
        ("Banglish", "ajke chorom ek shocking khobor ber holo"),
        ("English", "Scientists reveal shocking secret that changes everything")
    ]
    print(f"Vocab Size: {tokenizer.vocab_size:,}\n")
    for cat, text in test_phrases:
        toks = tokenizer.tokenize(text)
        print(f"  [{cat}] ({len(text.split())} words -> {len(toks)} tokens):")
        print(f"   Tokens: {' · '.join(toks)}\n")
else:
    print("Multi-head model uses custom character/word level vocabulary.")


## 6. Training Configuration, Auto-Resume & Extra Epochs
- **`RESUME_IF_INTERRUPTED`**: If Colab/Kaggle drops mid-training, keeps your checkpoint and resumes from the exact epoch where it stopped!
- **`EXTRA_EPOCHS`**: Add additional epochs anytime on top of an already completed run!


In [ ]:
# ==============================================================================
# 🎛️ TRAINING HYPERPARAMETERS & CONTINUATION SETTINGS
# ==============================================================================
TRAIN_EPOCHS = 2               # Base target epochs
EXTRA_EPOCHS = 0               # Increase to e.g. 2 to add extra training anytime!
RESUME_IF_INTERRUPTED = True   # Automatically picks up from latest checkpoint
BATCH_SIZE = 2                 # Batch size
LEARNING_RATE = 2e-4 if "llm" == "llm" else (3e-5 if "llm" == "seq2seq" else 1e-3)
# ==============================================================================


## 7. Model Training Execution


In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset

total_epochs = TRAIN_EPOCHS + EXTRA_EPOCHS
latest_ckpt = find_latest_checkpoint(OUTPUT_DIR) if (RESUME_IF_INTERRUPTED or EXTRA_EPOCHS > 0) else None
if latest_ckpt:
    print(f"🔄 Checkpoint found: Resuming training from {latest_ckpt} (Total Epochs: {total_epochs})")
else:
    print(f"🌱 Starting fresh training run for {total_epochs} epoch(s)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
tokenizer.padding_side = "right"

device_map = "auto" if gpu_count > 0 else None
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if PRECISION == "fp16" else torch.float32,
    device_map=device_map,
    trust_remote_code=True
)

targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] if any(k in MODEL_ID.lower() for k in ["qwen", "gemma", "llama", "tiger", "titu", "smol"]) else ["q_proj", "v_proj"]
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=targets, bias="none", task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

SYSTEM_PROMPT = "You are a news editor. Rewrite sensational clickbait into concise, factual, neutral titles."

def tokenize_fn(item):
    orig = item["original_title"].strip()
    body = (item.get("body") or "").strip()
    target = item["target_title"].strip()
    user_msg = f"Clickbait Headline: {orig}\nContext: {body[:300]}\nTask: Rewrite factual." if body else f"Clickbait Headline: {orig}\nTask: Rewrite factual."

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}, {"role": "assistant", "content": target}]
        full_text = tokenizer.apply_chat_template(msgs, tokenize=False)
        prompt_msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    else:
        prompt_text = f"### System:\n{SYSTEM_PROMPT}\n\n### User:\n{user_msg}\n\n### Assistant:\n"
        full_text = prompt_text + f"{target}{tokenizer.eos_token}"

    enc_full = tokenizer(full_text, max_length=256, truncation=True)
    enc_prompt = tokenizer(prompt_text, max_length=256, truncation=True)
    labels = list(enc_full["input_ids"])
    for i in range(min(len(enc_prompt["input_ids"]), len(labels))):
        labels[i] = -100
    return {"input_ids": enc_full["input_ids"], "attention_mask": enc_full["attention_mask"], "labels": labels}

train_ds = Dataset.from_pandas(data_bundle["df_train"]).map(tokenize_fn, remove_columns=list(data_bundle["df_train"].columns))
val_ds = Dataset.from_pandas(data_bundle["df_val"]).map(tokenize_fn, remove_columns=list(data_bundle["df_val"].columns))

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    num_train_epochs=total_epochs,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=(PRECISION == "fp16" and torch.cuda.is_available()),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True)
)

trainer.train(resume_from_checkpoint=latest_ckpt if (RESUME_IF_INTERRUPTED or EXTRA_EPOCHS > 0) else None)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Training finished! Model adapter saved to {OUTPUT_DIR}")


## 8. Evaluation Benchmark on Unseen Test Split


In [ ]:
import evaluate
rouge = evaluate.load("rouge")

test_df = data_bundle["df_test"].head(20)
predictions = []
references = test_df["target_title"].tolist()

print("Generating predictions on holdout test set...")
for _, r in test_df.iterrows():
    orig = r["original_title"]
    if "llm" == "llm":
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
            msgs = [{"role": "system", "content": "You are a news editor. Rewrite sensational clickbait into concise, factual, neutral titles."}, {"role": "user", "content": f"Clickbait Headline: {orig}\nTask: Rewrite factual."}]
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"### System:\n{SYSTEM_PROMPT}\n\n### User:\nClickbait: {orig}\nTask: Rewrite.\n\n### Assistant:\n"
        inp = tokenizer(prompt, return_tensors="pt")
        if hasattr(model, "device"):
            inp = {k: v.to(model.device) for k, v in inp.items()}
        with torch.no_grad():
            gen = model.generate(**inp, max_new_tokens=32, do_sample=False)
        pred = tokenizer.decode(gen[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    elif "llm" == "seq2seq":
        inp = tokenizer(f"rewrite: {orig}", return_tensors="pt")
        if hasattr(model, "device"):
            inp = {k: v.to(model.device) for k, v in inp.items()}
        with torch.no_grad():
            gen = model.generate(**inp, max_new_tokens=32)
        pred = tokenizer.decode(gen[0], skip_special_tokens=True).strip()
    else:
        pred = orig  # BiGRU baseline placeholder
    predictions.append(pred)

scores = rouge.compute(predictions=predictions, references=references)
df_eval = pd.DataFrame([
    {"Metric": "ROUGE-1", "Score": round(scores["rouge1"], 4)},
    {"Metric": "ROUGE-2", "Score": round(scores["rouge2"], 4)},
    {"Metric": "ROUGE-L", "Score": round(scores["rougeL"], 4)},
])
print("\n🏆 Evaluation Scores on Holdout Test Set:")
display(df_eval)

# Side-by-side predictions
df_samples = pd.DataFrame({
    "Input Clickbait": test_df["original_title"],
    "Model Prediction": predictions,
    "Ground Truth": references
}).head(5)
print("\n🔍 Sample Model Predictions:")
display(df_samples)


## 9. 🎁 Export Trained Weights to ZIP Archive
Compresses the trained model weights, configuration, and tokenizer from `tigerllm_1b_lora` into a single downloadable `.zip` file!


In [ ]:
archive_name = "tigerllm_1b_lora_weights"
zip_filename = f"{archive_name}.zip"

print(f"📦 Packaging weights from {OUTPUT_DIR} into {zip_filename}...")
shutil.make_archive(archive_name, "zip", OUTPUT_DIR)
zip_size_mb = os.path.getsize(zip_filename) / (1024**2)
print(f"✓ Archive ready: {zip_filename} ({zip_size_mb:.2f} MB)")

# Automatic Colab Download Trigger
if "google.colab" in sys.modules:
    try:
        from google.colab import files
        print("⬇️ Triggering automatic download in your browser...")
        files.download(zip_filename)
    except Exception as e:
        print(f"Colab download notice: {e}")
else:
    print(f"📂 Saved in working directory: {os.path.abspath(zip_filename)}")
    print(f"   (On Kaggle: download directly from the output panel on the right sidebar!)")
